In [1]:
import pandas as pd
import numpy as np
import torch
import lightning as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, NBeats
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import MAE, SMAPE
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:30: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
# ==========================================
# 1. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ (M4 Monthly)
# ==========================================
# Скачайте M4-Monthly.csv с https://github.com/Mcompetitions/M4-methods/tree/master/Dataset
df = pd.read_csv("archive/Monthly-train.csv")

# Если в файле нет колонки freq, можно задать вручную
if "freq" not in df.columns:
    df["freq"] = "Monthly"

# Преобразуем из широкого формата в длинный
id_col = "id" if "id" in df.columns else df.columns[0]
value_cols = [c for c in df.columns if c.startswith("V") or c.startswith("v")]
df_long = df.melt(id_vars=[id_col, "freq"], value_vars=value_cols, 
                  var_name="time_step", value_name="target")

# Удаляем NaN (в M4 хвосты иногда пустые)
df_long = df_long.dropna(subset=["target"])

# Создаём непрерывный time_idx внутри каждой серии
df_long["time_idx"] = df_long.groupby(id_col).cumcount()
df_long = df_long.rename(columns={id_col: "series_id"})

print(f"Структура: {len(df_long['series_id'].unique())} рядов, "
      f"длина ряда: {df_long['time_idx'].nunique()}")


C:\Users\dell\AppData\Local\Temp\ipykernel_4876\2417730217.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["freq"] = "Monthly"


Структура: 48000 рядов, длина ряда: 2794


In [3]:
# ==========================================
# 2. РАЗДЕЛЕНИЕ TRAIN / VAL ПО ВРЕМЕНИ
# ==========================================
max_idx = df_long["time_idx"].max()
prediction_length = 18  # горизонт для Monthly
encoder_length = 60     # длина истории (можно подобрать)

In [4]:
# 1. Жёсткая подготовка DataFrame
df_clean = df_long[["time_idx", "target", "series_id"]].copy()
df_clean["target"] = df_clean["target"].astype(float)          # таргет должен быть float
df_clean["series_id"] = df_clean["series_id"].astype(str)      # group_ids должны быть строками
df_clean["time_idx"] = df_clean["time_idx"].astype(int)        # время целое

# 2. Разделение
val_start = df_clean["time_idx"].max() - prediction_length
train_df = df_clean[df_clean["time_idx"] < val_start].copy()
val_df = df_clean[df_clean["time_idx"] >= (val_start - encoder_length)].copy()

# 3. Создание датасета с ЯВНЫМ отключением авто-детекта
training_dataset = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="target",
    group_ids=["series_id"],
    min_encoder_length=encoder_length,
    max_encoder_length=encoder_length,
    min_prediction_length=prediction_length,
    max_prediction_length=prediction_length,
    target_normalizer=GroupNormalizer(groups=["series_id"]),
    add_relative_time_idx=False,
    add_encoder_length=False,
    # 🔑 ЯВНО задаём пустые списки, чтобы time_idx НЕ попал в reals
    time_varying_known_reals=[],
    time_varying_unknown_reals=["target"],
    static_categoricals=[],
    static_reals=[],
    allow_missing_timesteps=False,
)

# 🔍 Проверка перед обучением (обязательно!)
print(f"✅ Reals: {training_dataset.reals}")
print(f"✅ Unknown reals: {training_dataset.time_varying_unknown_reals}")
print(f"✅ Categoricals: {training_dataset.flat_categoricals}")
assert len(training_dataset.reals) == 1 and training_dataset.reals[0] == "target", "❌ Ошибка: в reals попало лишнее!"

val_dataset = TimeSeriesDataSet.from_dataset(training_dataset, val_df, predict=False)
train_dl = training_dataset.to_dataloader(train=True, batch_size=64, num_workers=2)
val_dl = val_dataset.to_dataloader(train=False, batch_size=64, num_workers=2)

c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\pytorch_forecasting\data\timeseries\_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 10769 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__series_id': 'M10000'}, {'__group_id__series_id': 'M10001'}, {'__group_id__series_id': 'M10002'}, {'__group_id__series_id': 'M10003'}, {'__group_id__series_id': 'M10004'}, {'__group_id__series_id': 'M10005'}, {'__group_id__series_id': 'M10006'}, {'__group_id__series_id': 'M10007'}, {'__group_id__series_id': 'M10008'}, {'__group_id__series_id': 'M10009'}]
  warnings.warn(


✅ Reals: ['target']
✅ Unknown reals: ['target']
✅ Categoricals: []


c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\pytorch_forecasting\data\timeseries\_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 1 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__series_id': 'M36044'}]
  warnings.warn(


In [ ]:
# 4. ИНИЦИАЛИЗАЦИЯ NBeats И ОБУЧЕНИЕ
# ==========================================
early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=5, verbose=True)
checkpoint_callback = ModelCheckpoint(dirpath="checkpoints/", filename="nbeats_m4_monthly", save_top_k=1, monitor="val_loss")

trainer = pl.Trainer(
    max_epochs=50,
    accelerator="gpu",
    devices=1,
    callbacks=[early_stop_callback, checkpoint_callback],
    gradient_clip_val=0.1,
    #limit_train_batches=100,  # уберите для полного обучения
)

model = NBeats.from_dataset(
    training_dataset,
    learning_rate=1e-3,
    loss=MAE(),
    log_interval=10,
    log_val_interval=5,
    backcast_loss_ratio=1.0,
    widths=[32, 512],  # архитектура блоков
    num_blocks=[3, 3], # количество trend/seasonal блоков
)

# Обучение
trainer.fit(
    model,
    train_dataloaders=train_dl,
    val_dataloaders=val_dl,
)

# Загрузка лучшей модели
model = NBeats.load_from_checkpoint(checkpoint_callback.best_model_path)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'los

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Epoch 0: 100%|██████████| 105844/105844 [48:43<00:00, 36.20it/s, v_num=15, train_loss_step=349.0, val_loss=105.0]

Metric val_loss improved. New best score: 105.312


Epoch 1:   3%|▎         | 2729/105844 [01:13<45:58, 37.38it/s, v_num=15, train_loss_step=266.0, val_loss=105.0, train_loss_epoch=366.0]  


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [6]:
import torch
import numpy as np

# 1. Прогнозы (mode="prediction" уже вернул значения в исходных единицах)
y_pred = model.predict(val_dl, mode="prediction")
y_pred = y_pred.squeeze(-1).cpu().numpy()

# 2. Истинные значения: вручную собираем из батчей (гарантированно работает)
y_true_list = []
for batch in val_dl:
    x, y = batch          # y всегда кортеж: (target_tensor, weight_tensor)
    y_true_list.append(y[0])  # берём только target
y_true = torch.cat(y_true_list, dim=0).cpu().numpy()

# 3. Проверка совпадения размерностей
assert y_pred.shape == y_true.shape, f"❌ Shape mismatch: pred {y_pred.shape} vs true {y_true.shape}"

# 4. SMAPE в стиле M4
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    denominator = np.where(denominator == 0, 1e-8, denominator)
    return np.mean(2.0 * np.abs(y_true - y_pred) / denominator) * 100

print(f"✅ Validation SMAPE: {smape(y_true, y_pred):.2f}%")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'predict_dataloader' to speed up the dataloader worker initialization.


✅ Validation SMAPE: 22.03%


In [8]:
# ==========================================
# 3. ПРОГНОЗ + РАСЧЁТ SMAPE (без возни с индексами)
# ==========================================
res = model.predict(val_dl, mode="prediction", return_y=True)

# 1. Прогнозы
y_pred = res[0].squeeze(-1).cpu().numpy()  # (N_окон, prediction_length)

# 2. Истинные значения (y всегда последний элемент в кортеже вывода)
y_raw = res[-1]
y_true = y_raw[0].cpu().numpy() if isinstance(y_raw, tuple) else y_raw.cpu().numpy()

# ✅ Гарантированно совпадают!
assert y_pred.shape == y_true.shape, f"❌ Shape mismatch: {y_pred.shape} vs {y_true.shape}"

# Разворачиваем в 1D
y_pred_flat = y_pred.reshape(-1)
y_true_flat = y_true.reshape(-1)

print(f"📊 Успешно спрогнозировано точек: {len(y_pred_flat)}")
print(f"📉 Пропущено из-за короткой истории: {len(test_long) - len(y_pred_flat)} (это норма для M4)")

# ==========================================
# 4. SMAPE
# ==========================================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    denom = np.where(denom == 0, 1e-8, denom)
    return np.mean(2.0 * np.abs(y_true - y_pred) / denom) * 100

print(f"✅ Test SMAPE: {smape(y_true_flat, y_pred_flat):.2f}%")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


📊 Успешно спрогнозировано точек: 36


NameError: name 'test_long' is not defined

In [13]:
import lightning as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_forecasting import NBeats, GroupNormalizer
from pytorch_forecasting.metrics import MAE, SMAPE

# ==========================================
# 1. КОНФИГ ПО СТАТЬЕ (M4 Monthly)
# ==========================================
PREDICTION_LENGTH = 18          # официальный горизонт M4 Monthly
ENCODER_LENGTH = 36             # 2 сезона (статья рекомендует ≥ 2× horizon)
BATCH_SIZE = 64                 # в статье: 64-128
LR = 1e-3                       # Adam, lr=1e-3
BACKCAST_LOSS_RATIO = 0.1       # регуляризация backcast ветки (статья: 0.1)

# ==========================================
# 2. НОРМАЛИЗАЦИЯ (Local Scaling из статьи)
# ==========================================
target_normalizer = GroupNormalizer(
    groups=["series_id"],
    center=True,
    scale_by_group=True,
    transformation="softplus"   # защищает от деления на 0 при близких к 0 значениях
)

# ==========================================
# 3. DATASET (строго как в M4)
# ==========================================
training_dataset = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="target",
    group_ids=["series_id"],
    min_encoder_length=ENCODER_LENGTH,
    max_encoder_length=ENCODER_LENGTH,
    min_prediction_length=PREDICTION_LENGTH,
    max_prediction_length=PREDICTION_LENGTH,
    target_normalizer=target_normalizer,
    add_relative_time_idx=False,
    add_encoder_length=False,
    time_varying_known_reals=[],
    time_varying_unknown_reals=["target"],
    static_categoricals=[],
    static_reals=[],
    allow_missing_timesteps=False,
)

val_dataset = TimeSeriesDataSet.from_dataset(training_dataset, val_df, predict=False)
train_dl = training_dataset.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0, pin_memory=True)
val_dl = val_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0, pin_memory=True)

# ==========================================
# 4. МОДЕЛЬ (архитектура из статьи)
# ==========================================
model = NBeats.from_dataset(
    training_dataset,
    learning_rate=1e-3,
    loss=MAE(),
    log_interval=10,
    log_val_interval=5,
    # 🔑 Ключевые параметры Interpretable режима (Oreshkin et al., ICLR 2020)
    backcast_loss_ratio=0.0,              # Отключаем backcast-ошибку → модель учится только прогнозировать
    stack_types=["trend", "seasonality"], # Явные базисные функции (линейный тренд + ряд Фурье)
    num_blocks=[2, 2],                    # 2 блока тренда + 2 блока сезонности
    widths=[256, 256],                    # В статье для Interpretable используется 256 (не 512)                     # Полносвязных слоёв внутри каждого блока
)

# ==========================================
# 5. TRAINER (протокол обучения из статьи)
# ==========================================
early_stop = EarlyStopping(
    monitor="val_loss", 
    min_delta=1e-4, 
    patience=7, 
    verbose=True,
    mode="min"
)
checkpoint = ModelCheckpoint(
    dirpath="checkpoints/", 
    filename="nbeats_m4_monthly", 
    save_top_k=3,               # сохраняем топ-3 для ансамблирования
    monitor="val_loss"
)
lr_monitor = LearningRateMonitor(logging_interval="epoch")

trainer = pl.Trainer(
    max_epochs=100,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",       # mixed precision (статья не использует, но ускоряет без потери качества)
    gradient_clip_val=0.1,
    callbacks=[early_stop, checkpoint, lr_monitor],
    enable_progress_bar=True,
    log_every_n_steps=10,
)

# Обучение
trainer.fit(model, train_dataloaders=train_dl, val_dataloaders=val_dl)

# Загрузка лучшей модели
best_model = NBeats.load_from_checkpoint(checkpoint.best_model_path)

c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\pytorch_forecasting\data\timeseries\_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 15 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__series_id': 'M47808'}, {'__group_id__series_id': 'M47809'}, {'__group_id__series_id': 'M47810'}, {'__group_id__series_id': 'M47811'}, {'__group_id__series_id': 'M47812'}, {'__group_id__series_id': 'M47849'}, {'__group_id__series_id': 'M47850'}, {'__group_id__series_id': 'M47851'}, {'__group_id__series_id': 'M47853'}, {'__group_id__series_id': 'M47856'}]
  warnings.warn(
c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\pytorch_forecasting\data\timeseries\_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction l

Epoch 0:   2%|▏         | 2534/122476 [07:52<6:12:52,  5.36it/s, v_num=18, train_loss_step=374.0]
                                                                           

c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.
c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Epoch 0:   6%|▌         | 6981/122476 [03:24<56:23, 34.14it/s, v_num=19, train_loss_step=642.0]  


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [14]:
# ==========================================
# 0. ИМПОРТЫ & КОНФИГ (M4 Yearly)
# ==========================================
import pandas as pd
import numpy as np
import torch
import lightning as pl
from pytorch_forecasting import TimeSeriesDataSet, NBeats, GroupNormalizer
from pytorch_forecasting.metrics import MAE
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor

pl.seed_everything(42, workers=True)  # Воспроизводимость

# 📏 Гиперпараметры строго под M4 Yearly
PREDICTION_LENGTH = 6   # Официальный горизонт M4 Yearly
ENCODER_LENGTH = 12     # 2 × horizon (статья рекомендует ≥ 2 сезона)
BATCH_SIZE = 64
LR = 1e-3

Seed set to 42


In [17]:
# ==========================================
# 1. ЗАГРУЗКА & TRANSFORM (Wide → Long)
# ==========================================
def load_m4_yearly(path):
    df = pd.read_csv(path)
    id_col = "id" if "id" in df.columns else df.columns[0]
    value_cols = [c for c in df.columns if c.startswith("V") or c.startswith("v")]
    
    long = df.melt(id_vars=[id_col], value_vars=value_cols, 
                   var_name="time_step", value_name="target")
    long = long.dropna(subset=["target"])
    long["time_idx"] = long.groupby(id_col).cumcount()
    long = long.rename(columns={id_col: "series_id"})
    
    long["series_id"] = long["series_id"].astype(str)
    long["target"] = long["target"].astype(float)
    long["time_idx"] = long["time_idx"].astype(int)
    return long

train_long = load_m4_yearly("archive/Yearly-train.csv")
test_long  = load_m4_yearly("archive/Yearly-test.csv")

print(f"📊 Train: {train_long['series_id'].nunique()} серий, {len(train_long)} точек")
print(f"📊 Test:  {test_long['series_id'].nunique()} серий, {len(test_long)} точек")

📊 Train: 23000 серий, 720458 точек
📊 Test:  23000 серий, 138000 точек


In [30]:
# ==========================================
# 2. РАЗДЕЛЕНИЕ & DATASET
# ==========================================
max_time = train_long["time_idx"].max()
val_start = max_time - PREDICTION_LENGTH

train_df = train_long[train_long["time_idx"] < val_start].copy()
val_df   = train_long[train_long["time_idx"] >= val_start - ENCODER_LENGTH].copy()

training_dataset = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx", target="target", group_ids=["series_id"],
    min_encoder_length=ENCODER_LENGTH, max_encoder_length=ENCODER_LENGTH,
    min_prediction_length=PREDICTION_LENGTH, max_prediction_length=PREDICTION_LENGTH,
    target_normalizer=GroupNormalizer(groups=["series_id"], center=True, scale_by_group=True, transformation="softplus"),
    add_relative_time_idx=False, add_encoder_length=False,
    time_varying_known_reals=[], time_varying_unknown_reals=["target"],
    static_categoricals=[], static_reals=[], allow_missing_timesteps=False,
)

val_dataset = TimeSeriesDataSet.from_dataset(training_dataset, val_df, predict=False)
train_dl = training_dataset.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=4, pin_memory=True)
val_dl   = val_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=4, pin_memory=True)

c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\pytorch_forecasting\data\timeseries\_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 3939 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__series_id': 'Y10042'}, {'__group_id__series_id': 'Y10082'}, {'__group_id__series_id': 'Y10111'}, {'__group_id__series_id': 'Y10135'}, {'__group_id__series_id': 'Y10136'}, {'__group_id__series_id': 'Y10145'}, {'__group_id__series_id': 'Y10183'}, {'__group_id__series_id': 'Y10185'}, {'__group_id__series_id': 'Y10186'}, {'__group_id__series_id': 'Y102'}]
  warnings.warn(
c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\pytorch_forecasting\data\timeseries\_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction l

In [31]:
# ==========================================
# 3. МОДЕЛЬ (Interpretable, Yearly-адаптация)
# ==========================================
model = NBeats.from_dataset(
    training_dataset,
    learning_rate=LR, loss=MAE(),
    log_interval=10, log_val_interval=5,
    # 🔑 Interpretable Yearly
    backcast_loss_ratio=0.0,          # Учимся только на forecast
    stack_types=["trend", "seasonality"],
    num_blocks=[4, 2],                # Yearly: сильный тренд, слабая сезонность
    widths=[256, 256],                # Стандарт для Interpretable
)

c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [32]:
# ==========================================
# 4. TRAINER & ОБУЧЕНИЕ
# ==========================================
early_stop = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=7, mode="min")
checkpoint = ModelCheckpoint(dirpath="checkpoints_yearly/", save_top_k=3, monitor="val_loss")
lr_monitor = LearningRateMonitor(logging_interval="epoch")

trainer = pl.Trainer(
    max_epochs=2,
    accelerator="gpu", devices=1, precision="16-mixed",
    gradient_clip_val=0.1,
    callbacks=[early_stop, checkpoint, lr_monitor],
)

trainer.fit(model, train_dataloaders=train_dl, val_dataloaders=val_dl)
best_model = NBeats.load_from_checkpoint(checkpoint.best_model_path)
print("✅ Обучение завершено. Лучший чекпоинт загружен.")

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory C:\Users\dell\Desktop\Учёба + ШАД\ML\Проект\checkpoints_yearly exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\utilities\model_summary\model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name            | Type       | Params | Mode  | FLOPs
----------------------------------------

Epoch 0:   2%|▏         | 82/5256 [02:27<2:35:14,  0.56it/s, v_num=25, train_loss_step=615.0]
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Epoch 1: 100%|██████████| 5256/5256 [01:49<00:00, 47.86it/s, v_num=27, train_loss_step=433.0, val_loss=1.36e+3, train_loss_epoch=445.0]

`Trainer.fit` stopped: `max_epochs=2` reached.


Epoch 1: 100%|██████████| 5256/5256 [01:49<00:00, 47.82it/s, v_num=27, train_loss_step=433.0, val_loss=1.36e+3, train_loss_epoch=445.0]


c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


✅ Обучение завершено. Лучший чекпоинт загружен.


In [33]:
# ==========================================
# 5. ТЕСТОВАЯ ОЦЕНКА (Robust, без индекс-багов)
# ==========================================
# 5.1. Сдвиг time_idx теста + склейка с историей
max_train_time = train_long.groupby("series_id")["time_idx"].max().reset_index()
max_train_time.columns = ["series_id", "max_t"]

# Оставляем только серии, которые были в тренировке (стандарт M4)
test_shifted = test_long[test_long["series_id"].isin(train_long["series_id"])].copy()
test_shifted = test_shifted.merge(max_train_time, on="series_id")
test_shifted["time_idx"] = test_shifted["time_idx"] + test_shifted["max_t"] + 1
test_shifted = test_shifted.drop(columns=["max_t"])

# Берём последние ENCODER_LENGTH точек из тренировки для каждой серии
history = train_long.groupby("series_id").tail(ENCODER_LENGTH).copy()

eval_df = pd.concat([history, test_shifted], ignore_index=True)
eval_df = eval_df.sort_values(["series_id", "time_idx"]).reset_index(drop=True)

# 5.2. Dataset & DataLoader для eval
eval_dataset = TimeSeriesDataSet.from_dataset(training_dataset, eval_df, predict=False)
eval_dl = eval_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0, pin_memory=True)

# 5.3. Прогноз & метрика
res = best_model.predict(eval_dl, mode="prediction", return_y=True)
y_pred = res[0].squeeze(-1).cpu().numpy()
y_true_raw = res[-1]
y_true = y_true_raw[0].cpu().numpy() if isinstance(y_true_raw, tuple) else y_true_raw.cpu().numpy()

assert y_pred.shape == y_true.shape, f"❌ Shape mismatch: {y_pred.shape} vs {y_true.shape}"

y_pred_flat = y_pred.reshape(-1)
y_true_flat = y_true.reshape(-1)

def smape(y_true, y_pred):
    d = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    d = np.where(d == 0, 1e-8, d)
    return np.mean(200 * np.abs(y_true - y_pred) / d)

print(f"✅ Yearly Test SMAPE: {smape(y_true_flat, y_pred_flat):.2f}%")
print(f"📉 Спрогнозировано точек: {len(y_pred_flat)} (короткие ряды отфильтрованы автоматически)")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\dell\Desktop\Учёба + ШАД\venv\.venv_ml_project\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


✅ Yearly Test SMAPE: 43.77%
📉 Спрогнозировано точек: 138000 (короткие ряды отфильтрованы автоматически)
